# Tutorial 06: Custom Verses

**Time**: 60 minutes | **Difficulty**: Intermediate-Advanced | **Prerequisites**: Tutorials 01-05

---

## What You'll Learn
- The contract every verse must satisfy
- How to build a minimal working verse from scratch
- How reward design shapes agent behaviour
- How to register and train on your custom verse

---

## The Verse Contract

Every Multiverse environment (a "verse") must implement four methods:

```
seed(seed)   → set the random seed for reproducibility
reset()      → start a new episode, return initial observation
step(action) → apply action, return (obs, reward, done, truncated, info)
```

Plus two properties:
```
observation_space  → describes the shape/type of observations
action_space       → describes valid actions
```

That's it. If your class satisfies this contract, Multiverse can train any agent on it.

---

## Setup

In [ ]:
import subprocess
import sys
import os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    print(output if output.strip() else '(no output)')
    return result.returncode == 0

# Make sure we can import from the project root
project_root = os.path.abspath('../..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print('Project root:', project_root)

## Part 1: Read an Existing Verse

Before building one, let's understand the structure by reading `line_world.py`.
It's the simplest verse in the library.

In [ ]:
# Read the first ~80 lines to see the structure
with open('verses/line_world.py') as f:
    lines = f.readlines()

print(f'Total lines: {len(lines)}')
print()
print('--- First 80 lines ---')
print(''.join(lines[:80]))

## Part 2: List All Available Verses

Before building your own, see what already exists. You might be able to adapt one.

In [ ]:
run('multiverse universe list')

## Part 3: Build a Minimal Custom Verse

We'll build **Temperature World**: a thermostat problem.

- State: current temperature (0–100)
- Target: 70 degrees
- Actions: 0=cool (-5), 1=hold, 2=heat (+5)
- Reward: closer to 70 = better
- Done: within 2 degrees of target, or 50 steps

This is a simple but non-trivial verse: the agent must learn to stop overshooting.

In [ ]:
import random


class TemperatureWorld:
    """A thermostat control problem.
    
    Observation: {temp: int, target: int, dist: int, t: int}
    Actions: 0=cool (-5 deg), 1=hold, 2=heat (+5 deg)
    Reward: -abs(temp - target) / 100.0 each step; +1.0 on reaching target zone
    Done: within 2 degrees of target, or max_steps reached
    """

    TARGET = 70
    TOLERANCE = 2
    MAX_STEPS = 50

    def __init__(self):
        self._rng = random.Random()
        self._temp = 50
        self._t = 0
        self._done = False

        # Describe spaces as plain dicts for this standalone example
        self.observation_space = {'type': 'dict', 'keys': ['temp', 'target', 'dist', 't']}
        self.action_space = {'type': 'discrete', 'n': 3, 'notes': '0=cool, 1=hold, 2=heat'}

    def seed(self, seed=None):
        self._rng = random.Random(seed)

    def reset(self):
        # Start at a random temperature between 20 and 120
        self._temp = self._rng.randint(20, 120)
        self._t = 0
        self._done = False
        return self._obs()

    def step(self, action):
        if self._done:
            return self._obs(), 0.0, True, False, {'warning': 'called after done'}

        # Apply action
        delta = {0: -5, 1: 0, 2: 5}[int(action)]
        self._temp = max(0, min(100, self._temp + delta))
        self._t += 1

        dist = abs(self._temp - self.TARGET)

        # Reward: negative distance + bonus for reaching target
        if dist <= self.TOLERANCE:
            reward = 1.0
            self._done = True
        else:
            reward = -dist / 100.0  # small step penalty proportional to distance

        truncated = (self._t >= self.MAX_STEPS) and not self._done
        if truncated:
            self._done = True

        return self._obs(), reward, self._done, truncated, {'temp': self._temp, 'dist': dist}

    def _obs(self):
        dist = abs(self._temp - self.TARGET)
        return {'temp': self._temp, 'target': self.TARGET, 'dist': dist, 't': self._t}


print('TemperatureWorld defined.')
print('Observation space:', TemperatureWorld().observation_space)
print('Action space:', TemperatureWorld().action_space)

## Part 4: Test the Verse Manually

Always test your verse with a manual rollout before training on it.
This catches logic bugs (wrong reward sign, episodes that never end, etc.).

In [ ]:
env = TemperatureWorld()
env.seed(42)

obs = env.reset()
print(f'Initial: {obs}')
print()

total_reward = 0
for step in range(20):
    # Simple rule: cool if too hot, heat if too cold, hold if close
    if obs['temp'] > env.TARGET + env.TOLERANCE:
        action = 0  # cool
    elif obs['temp'] < env.TARGET - env.TOLERANCE:
        action = 2  # heat
    else:
        action = 1  # hold

    obs, reward, done, truncated, info = env.step(action)
    total_reward += reward
    print(f'Step {step+1:2d}: action={["cool","hold","heat"][action]}, temp={obs["temp"]:3d}, reward={reward:+.3f}')

    if done:
        status = 'GOAL' if reward > 0 else 'TRUNCATED'
        print(f'\nEpisode ended: {status}  |  total reward: {total_reward:.3f}')
        break

## Part 5: Reward Design Matters

The reward function you choose profoundly shapes what the agent learns.
Let's demonstrate two common mistakes:

**Sparse reward**: agent only gets +1 on success, 0 otherwise → very hard to learn

**Dense reward**: agent gets a signal every step proportional to progress → much easier

Our TemperatureWorld uses dense reward (`-dist/100` per step). Let's compare.

In [ ]:
# Simulate a random agent under dense vs sparse reward
import random

def simulate_random(sparse=False, episodes=100, seed=1):
    rng = random.Random(seed)
    returns = []
    for _ in range(episodes):
        env = TemperatureWorld()
        env.seed(rng.randint(0, 9999))
        obs = env.reset()
        ep_return = 0
        for _ in range(50):
            action = rng.randint(0, 2)
            obs, reward, done, truncated, _ = env.step(action)
            if sparse:
                # Only reward on goal
                ep_return += reward if reward > 0 else 0
            else:
                ep_return += reward
            if done or truncated:
                break
        returns.append(ep_return)
    return returns

dense_returns = simulate_random(sparse=False)
sparse_returns = simulate_random(sparse=True)

print(f'Dense reward  — average return: {sum(dense_returns)/len(dense_returns):.3f}')
print(f'Sparse reward — average return: {sum(sparse_returns)/len(sparse_returns):.3f}')
print()
print('Dense reward gives the agent more signal to learn from.')
print('Sparse reward means the agent may never discover the goal at all.')

## Part 6: Write Your Verse to a File

To train on your verse with the Multiverse CLI, save it to `verses/` and register it.

In [ ]:
# Show the verse file skeleton — copy this to verses/temperature_world.py
skeleton = '''
# verses/temperature_world.py
# Save this file, then add it to verses/registry.py to use it with the CLI.

from core.types import SpaceSpec, VerseSpec
from core.verse_base import ResetResult, StepResult, Verse
import random

class TemperatureWorldVerse(Verse):
    TARGET = 70
    TOLERANCE = 2
    MAX_STEPS = 50

    def __init__(self, spec: VerseSpec) -> None:
        self.spec = spec
        self._rng = random.Random()
        self._temp = 50
        self._t = 0
        self._done = False
        self.observation_space = SpaceSpec(
            type="dict",
            keys=["temp", "target", "dist", "t"],
            subspaces={
                "temp":   SpaceSpec(type="vector", shape=(1,), dtype="int32"),
                "target": SpaceSpec(type="vector", shape=(1,), dtype="int32"),
                "dist":   SpaceSpec(type="vector", shape=(1,), dtype="int32"),
                "t":      SpaceSpec(type="vector", shape=(1,), dtype="int32"),
            },
        )
        self.action_space = SpaceSpec(type="discrete", n=3, notes="0=cool, 1=hold, 2=heat")

    def seed(self, seed=None):
        self._rng = random.Random(seed)

    def reset(self) -> ResetResult:
        self._temp = self._rng.randint(20, 120)
        self._t = 0
        self._done = False
        return ResetResult(obs=self._obs(), info={})

    def step(self, action) -> StepResult:
        delta = {0: -5, 1: 0, 2: 5}[int(action)]
        self._temp = max(0, min(100, self._temp + delta))
        self._t += 1
        dist = abs(self._temp - self.TARGET)
        if dist <= self.TOLERANCE:
            reward, done = 1.0, True
        else:
            reward, done = -dist / 100.0, False
        truncated = (self._t >= self.MAX_STEPS) and not done
        if truncated:
            done = True
        self._done = done
        return StepResult(obs=self._obs(), reward=reward, done=done,
                          truncated=truncated, info={"dist": dist})

    def _obs(self):
        return {"temp": self._temp, "target": self.TARGET,
                "dist": abs(self._temp - self.TARGET), "t": self._t}
'''

print(skeleton)

## Reward Design Cheatsheet

| Reward type | When to use | Watch out for |
|---|---|---|
| Sparse (+1 on goal) | Simple tasks where goal is easy to reach | Agent never finds the reward |
| Dense (distance-based) | Tasks where progress is measurable | Reward hacking (exploit shortcuts) |
| Shaped (potential-based) | When you know the value function | Must be properly potential-based |
| Step penalty | Encourage efficiency | Too large → agent gives up early |

**Most common mistake**: sparse reward in a large environment. Add at least a small
step penalty or distance-based shaping to give the agent signal.

---

## Next

- [Tutorial 07: Advanced Training — curriculum learning and safety](07_advanced_training.ipynb)
- [Verse Catalog](../reference/VERSE_CATALOG.md) — 25+ built-in environments to study